In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from collections import defaultdict
import math
import statsmodels.formula.api as smf

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default") 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA, TruncatedSVD
import torch
import category_encoders as ce



In [24]:

import sys
!{sys.executable} -m pip install category_encoders


  Obtaining dependency information for category_encoders from https://files.pythonhosted.org/packages/a6/06/afcae4dab08612dac244ace7f478543f4fb83bea94177231ef9b4f7bfa06/category_encoders-2.9.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: /ihome/xli/sek188/MSTHESIS/envs/.venv/bin/python -m pip install --upgrade pip


In [20]:
ldf = pd.read_csv('../week23/modC_pred_scores')

'followed news outlets' seems to be a very valuable data point, but given that it's in list form it's a bit hard to work with. We did try creating an embedding for it, but it does not reduce very well, so we save this for just the model. Also dropping native english speaker, since not significant.

In [28]:
cat_vars = ['type','topic', 'outlet', 'gender','news_check_frequency', 'survey_completed',
            'poli_leaning_binned', 'age_binned','education_binned', 'num_followed_outlets']
cont_vars = ['emb_pca_1', 'emb_pca_2','emb_pca_3', 'emb_pca_4', 'emb_pca_5']
label_col = "label_binary"

X = ldf[cat_vars + cont_vars]
y = ldf[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

encoder = ce.TargetEncoder(
    cols=cat_vars,
    smoothing=0.3,
    min_samples_leaf=10
)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

pipe = Pipeline(steps=[
    ("encode", encoder),
    ("model", rf)
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


              precision    recall  f1-score   support

           0       0.58      0.51      0.54      1425
           1       0.70      0.75      0.72      2130

    accuracy                           0.65      3555
   macro avg       0.64      0.63      0.63      3555
weighted avg       0.65      0.65      0.65      3555

ROC-AUC: 0.7022958570134256


In [ ]:
param_grid = {
    'model__max_depth': [10, 15, 20, 30, None],
    'model__min_samples_leaf': [1, 5, 10, 20],
    'model__n_estimators': [200, 500, 1000],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print(grid.best_params_)
print(classification_report(y_test, grid.predict(X_test)))
print("ROC-AUC:", roc_auc_score(y_test, grid.predict_proba(X_test)[:,1]))

In [31]:
from sklearn.calibration import CalibratedClassifierCV

# option 1 - adjust threshold
y_prob = grid.predict_proba(X_test)[:, 1]
threshold = 0.4  # lower threshold = more class 0 predictions
y_pred_adj = (y_prob >= threshold).astype(int)
print(classification_report(y_test, y_pred_adj))

              precision    recall  f1-score   support

           0       0.63      0.38      0.47      1425
           1       0.67      0.85      0.75      2130

    accuracy                           0.66      3555
   macro avg       0.65      0.61      0.61      3555
weighted avg       0.65      0.66      0.64      3555

